# ⚡ SpiceWizard × AMD Instinct MI300X
### Simulation-verified netlist generation — the generation half runs HERE, on AMD silicon

| | |
|---|---|
| **GPU** | AMD Instinct **MI300X** — 192 GB HBM3, ROCm |
| **Model** | `Qwen/Qwen2.5-Coder-32B-Instruct` — open weights, fp16, **fully GPU-resident** |
| **Stack** | PyTorch on ROCm (HuggingFace transformers) |
| **Role in the loop** | `RETRIEVE →` **`ADAPT (this notebook, on AMD)`** `→ VERIFY (LTspice, workstation)` |

**The pitch:** LLMs hallucinate circuits that look right. Ours has to prove it — every netlist
generated here goes to a real LTspice simulation and is measured against the spec before
anyone trusts it. The MI300X's 192 GB lets a 32B coder model (~62 GB fp16) sit entirely in
VRAM with room to spare — no quantization, no CPU offload, no compromises.

**Demo tip:** keep a terminal open with `watch -n 1 rocm-smi` while the generation cells run —
GPU% and VRAM% spiking on AMD hardware is the proof shot.

## 0 · Environment — what silicon are we on?

In [ ]:
import torch, platform

print("=" * 62)
print("  SpiceWizard generation node — environment")
print("=" * 62)
print(f"  Python          : {platform.python_version()}")
print(f"  PyTorch         : {torch.__version__}")
print(f"  ROCm (HIP)      : {torch.version.hip}")
print(f"  GPU             : {torch.cuda.get_device_name(0)}")
props = torch.cuda.get_device_properties(0)
print(f"  VRAM            : {props.total_memory / 1e9:.0f} GB")
print("=" * 62)

In [ ]:
import os
# Reliable-first downloads: skip HF's CDN accelerators (hf_transfer + Xet) —
# they 403 on some cloud IPs. Plain CDN is fine. Must be set BEFORE
# huggingface_hub is imported.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"]        = "1"

!pip -q install -U huggingface_hub transformers accelerate safetensors ipywidgets
print("deps ready")

## 1 · Load Qwen2.5-Coder-32B — downloads ONCE, then loads from disk

Weights persist in `~/models/` (NOT `/tmp`, which is wiped on instance restart).
`device_map={"": 0}` forces the **entire** model onto the MI300X — no CPU offload,
which 192 GB makes trivial for a 32B fp16 model.

In [ ]:
import glob, time
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
local_dir = os.path.expanduser(f"~/models/{MODEL_NAME.split('/')[-1].lower()}")

if not glob.glob(os.path.join(local_dir, "*.safetensors")):
    print("Downloading (first time only)...")
    snapshot_download(repo_id=MODEL_NAME, local_dir=local_dir)
else:
    print("Weights already on disk — skipping download.")

t0 = time.time()
tok = AutoTokenizer.from_pretrained(local_dir)
model = AutoModelForCausalLM.from_pretrained(
    local_dir, dtype=torch.float16, device_map={"": 0}   # whole model on the MI300X
)

used = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nLoaded in {time.time()-t0:.0f} s on: {next(model.parameters()).device}")
print(f"Model resident in VRAM: {used:.1f} GB of {total:.0f} GB "
      f"({100*used/total:.0f}% — fp16, zero offload)")

## 2 · Helpers — instrumented generation (every call reports tokens/sec on the MI300X)

In [ ]:
import re

SYSTEM_PROMPT = "You are an expert analog applications engineer who writes LTspice netlists."

_FENCE_RE = re.compile(r"```[^\n]*\r?\n(.*?)```", re.DOTALL)

def extract_netlist(text):
    """Strip markdown fences / prose; return through the final .end line."""
    m = _FENCE_RE.search(text)
    lines = (m.group(1) if m else text).strip().splitlines()
    if not m:
        start = next((i for i, l in enumerate(lines) if l.lstrip().startswith("*")), 0)
        lines = lines[start:]
    end = max((i for i, l in enumerate(lines) if l.strip().lower() in (".end", ".ends")), default=None)
    if end is not None:
        lines = lines[: end + 1]
    return "\n".join(lines).strip() + "\n"

def ask_qwen(user_prompt, max_new_tokens=900):
    """One instrumented generation on the MI300X: returns text, prints tok/s."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    n_new = out.shape[1] - inputs.input_ids.shape[1]
    print(f"[MI300X] {n_new} tokens in {dt:.1f} s  →  {n_new/dt:.1f} tok/s")
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def adapt_prompt(template, ic, spec_text):
    """Same contract the agent's verify loop enforces (generate_verify.py)."""
    return f"""Below is a KNOWN-WORKING LTspice netlist template for the {ic}.

Modify ONLY the passive component values (resistors, capacitors, source
amplitudes/frequencies) needed to meet the NEW SPEC below.
Do NOT change the {ic} subcircuit call line's pin order or node names.
Do NOT change the .lib reference or the analysis (.tran/.ac) directives.
Do NOT rename any nodes.

Output the complete modified netlist and nothing else.

TEMPLATE:
{template}

NEW SPEC:
{spec_text}
"""

def save_netlist(fname, text):
    open(fname, "w").write(text)
    print(f"saved → {fname}   (download via the file browser, verify on the workstation)")

print("helpers ready")

## Demo 1 · Spec adaptation — the hero case (AD8092, gain 2 → 5)

Given a known-good gain-2 template, Qwen must change **only** the feedback resistor
to hit gain 5, keeping the test signal in the part's linear range.
**Expected edit:** `R2 1K → 4K` (gain = 1 + R2/R1).

**Verify on the workstation:**
`python report_netlist.py AD8092_gain5.net --metric gain_db=13.98:1.0` → PASS ≈ 13.95 dB

In [ ]:
AD8092_TEMPLATE = """* AD8092 non-inverting amplifier (known-good template, gain = 2)
X§U1 IN N001 +V -V OUT AD8092
Vin IN 0 SINE(0 0.1 1Meg)
R1 N001 0 1K
R2 OUT N001 1K
Rload OUT 0 2K
V1 +V 0 5
V2 -V 0 -5
.tran 20u
.lib ADI.lib
.backanno
.end
"""

netlist = extract_netlist(ask_qwen(adapt_prompt(
    AD8092_TEMPLATE, "AD8092",
    "Non-inverting amplifier, closed-loop gain of 5 V/V (~13.98 dB), +/-5 V supplies. "
    "Keep the 100 mV / 1 MHz test signal (well inside the part's linear range).")))
print(netlist)
save_netlist("AD8092_gain5.net", netlist)

## Demo 2 · Generalization — different part, different family (AD811 video amp, gain 2 → 4)

**Expected edit:** feedback resistor `649 → ~1.95K` (gain = 1 + R2/R1).

**Verify:** `python report_netlist.py AD811_gain4.net --metric gain_db=12.04:1.0` → PASS ≈ 12 dB

In [ ]:
AD811_TEMPLATE = """* AD811 video amplifier (known-good template, gain = 2)
Vin IN 0 SINE(0 0.1 1Meg)
R1 N001 0 649
R2 OUT N001 649
X§U1 IN N001 +V -V OUT AD811
Rload OUT 0 150
V1 +V 0 15
V2 -V 0 -15
.tran 20u
.lib AD811.lib
.backanno
.end
"""

netlist = extract_netlist(ask_qwen(adapt_prompt(
    AD811_TEMPLATE, "AD811",
    "Non-inverting gain of 4 V/V (~12.04 dB), +/-15 V supplies, 150 ohm load, "
    "keep the 100 mV / 1 MHz test signal.")))
print(netlist)
save_netlist("AD811_gain4.net", netlist)

## Demo 3 · The physics trap — textbook-correct and physically impossible

Same gain-5 ask, but the spec **pins the test signal at 1 V / 10 MHz**. The correct-looking
resistor edit demands ~314 V/µs of slew rate; the real AD8092 delivers ~170. The netlist
below will *look* perfect — **and the simulator will fail it** (~10.3 dB measured, not 13.98).

That failure is the product: **only a simulator asks physics.**

**Verify (expect FAIL):** `python report_netlist.py AD8092_slewtrap.net --metric gain_db=13.98:1.0`

In [ ]:
SLEWTRAP_TEMPLATE = AD8092_TEMPLATE.replace("SINE(0 0.1 1Meg)", "SINE(0 1 10Meg)").replace(".tran 20u", ".tran 1u")

netlist = extract_netlist(ask_qwen(adapt_prompt(
    SLEWTRAP_TEMPLATE, "AD8092",
    "Non-inverting amplifier, closed-loop gain of 5 V/V (~13.98 dB), +/-5 V supplies. "
    "Keep the existing 1 V amplitude, 10 MHz test signal exactly as is.")))
print(netlist)
save_netlist("AD8092_slewtrap.net", netlist)

## Demo 4 · Combine two circuits — amp stage + RC low-pass into one netlist

Both blocks arrive using the same node names (`IN`/`OUT`) — Qwen must chain them,
resolve the collision, and emit ONE clean netlist with a single analysis.

**Verify:** `python report_netlist.py amp_lpf_chain.net --metric gain_db=6.0:1.0 --metric bandwidth_hz=15.9e3:5e3`
(gain-2 amp ≈ 6 dB in the passband; RC corner 1k/10n ≈ 15.9 kHz)

In [ ]:
AMP_STAGE = """* Stage A: AD8092 non-inverting amp, gain = 2
X§U1 IN N001 +V -V OUT AD8092
Vin IN 0 AC 1
R1 N001 0 1K
R2 OUT N001 1K
V1 +V 0 5
V2 -V 0 -5
.ac dec 100 10 10Meg
.lib ADI.lib
.end
"""

RC_LPF = """* Stage B: RC low-pass, corner ~15.9 kHz
R3 IN OUT 1K
C1 OUT 0 10n
.ac dec 100 10 10Meg
.end
"""

combine_prompt = f"""Combine the two LTspice circuit blocks below into ONE netlist that
implements: signal source → AD8092 amplifier (gain 2) → RC low-pass filter → node OUT.

Rules:
- The final output node of the whole chain must be named OUT, the source input node IN.
- Rename any colliding internal node names so the stages connect correctly (no shorts).
- Keep the AD8092 subcircuit call's pin order and the .lib ADI.lib reference unchanged.
- Exactly ONE analysis directive: .ac dec 100 10 10Meg
- Output the complete combined netlist and nothing else.

BLOCK A:
{AMP_STAGE}
BLOCK B:
{RC_LPF}"""

netlist = extract_netlist(ask_qwen(combine_prompt, max_new_tokens=700))
print(netlist)
save_netlist("amp_lpf_chain.net", netlist)

## Demo 5 · From-scratch generation — no template at all

**Verify:** `python report_netlist.py rc_lpf_1k.net --metric bandwidth_hz=1000:250` → −3 dB corner ≈ 1 kHz

In [ ]:
scratch_prompt = ("Write a complete LTspice netlist for a first-order passive RC low-pass filter "
                  "with a -3 dB cutoff of exactly 1 kHz. Input node IN driven by 'Vin IN 0 AC 1', "
                  "output node OUT. Include '.ac dec 100 1 100k' and '.end'. "
                  "Output only the netlist.")

netlist = extract_netlist(ask_qwen(scratch_prompt, max_new_tokens=300))
print(netlist)
save_netlist("rc_lpf_1k.net", netlist)

## Demo 6 · Throughput — batch generation for the data flywheel

Five spec variants generated back-to-back on the MI300X. On the workstation, each
verified pass is appended to `verified_pairs.jsonl` — a simulation-verified dataset
that grows with normal use (the RSFT flywheel).

In [ ]:
batch_specs = [
    ("AD8092_gain3.net",  "gain of 3 V/V (~9.54 dB)",   "gain_db=9.54:1.0"),
    ("AD8092_gain8.net",  "gain of 8 V/V (~18.06 dB)",  "gain_db=18.06:1.0"),
    ("AD8092_gain10.net", "gain of 10 V/V (~20.0 dB)",  "gain_db=20.0:1.0"),
    ("AD811_gain3.net",   "gain of 3 V/V (~9.54 dB)",   "gain_db=9.54:1.0"),
    ("AD811_gain6.net",   "gain of 6 V/V (~15.56 dB)",  "gain_db=15.56:1.0"),
]

t_all = time.time()
for fname, gain_txt, metric in batch_specs:
    ic = "AD8092" if fname.startswith("AD8092") else "AD811"
    tmpl = AD8092_TEMPLATE if ic == "AD8092" else AD811_TEMPLATE
    spec = (f"Non-inverting amplifier, closed-loop {gain_txt}, keep the 100 mV / 1 MHz "
            "test signal and supplies as in the template.")
    print(f"\n--- {fname} ({ic}: {gain_txt})")
    save_netlist(fname, extract_netlist(ask_qwen(adapt_prompt(tmpl, ic, spec))))

print(f"\n[MI300X] batch of {len(batch_specs)} candidate circuits in {time.time()-t_all:.0f} s")
print("verify each on the workstation:")
for fname, _, metric in batch_specs:
    print(f"  python report_netlist.py {fname} --metric {metric}")

## → Hand-off to the verification agent (workstation)

1. **Download** the generated `.net` files (right-click in the file browser → Download).
2. **Verify each one** in the agent repo (real LTspice, measured numbers, PASS/FAIL with margin):
   ```bash
   python report_netlist.py AD8092_gain5.net    --metric gain_db=13.98:1.0        # PASS ~13.95 dB
   python report_netlist.py AD811_gain4.net     --metric gain_db=12.04:1.0        # PASS ~12 dB
   python report_netlist.py AD8092_slewtrap.net --metric gain_db=13.98:1.0        # FAIL ~10.3 dB — the demo moment
   python report_netlist.py amp_lpf_chain.net   --metric gain_db=6.0:1.0 --metric bandwidth_hz=15.9e3:5e3
   python report_netlist.py rc_lpf_1k.net       --metric bandwidth_hz=1000:250
   ```
3. Or paste any netlist into the GUI's **Verify Spec** tab for the visual PASS/FAIL report.

**Division of labor:** the MI300X did the intelligence; LTspice does the physics; nothing
reaches a human until the physics agrees. Generated on AMD — proven in SPICE. ⚡